# Creating a  SQLite database with Python
SQLite is a relational database management system based on the SQL language optimized for use in small environments such as mobile apps. It does not require a separate server process to be run as needed in large database engines such as MySQL and Oracle. It's integrated with Python using a module called sqlite3.

In [86]:
import sqlite3
from pathlib import Path

### Create a SQLite database
We can start by creating the database file `student.db`.

In [87]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [88]:
## finding path
# !ls drive/MyDrive/SJSU/SJSU_Spring2026/CS22B/Activites/Module08_SQL

In [89]:
gdrive='drive/MyDrive/SJSU/SJSU_Spring2026/CS22B/Activites/Module08_SQL/'
Path(gdrive+'student.db').touch()

## Define the relationships between your data

Let's create a database to store the information about students and the courses that they enroll in.

- The `student` table that stores student information.
- The `course` table that stores course information.
- The `enrollment` table that stores the relationship between students and courses.

The following Entity Relationship Diagram (ERD) illustrates tables:student, course, and enrollment.
You can create it in Lucidchart (SJSU has license for students at SJSU). You can find it in [one.sjsu.edu](https://one.sjsu.edu)

<img src="https://www.sjsu.edu/people/wendy.lee/pics/CS122/ERD_school.png" width=450>
<br>
<img src="https://www.sjsu.edu/people/wendy.lee/pics/CS122/ERD_Cardinality.png" width=300>


[Learn more](https://www.guru99.com/database-normalization.html) about designing your relationship database tables to eliminate data redundancy and improve data integrity.

#### SQL command
You can export the SQL command from the Lucidchart ERD diagram tool but you will need to modify it to be accepted by SQLite.

`CREATE TABLE student (
  student_id INTEGER,
  firstname TEXT,
  lastname TEXT,
  PRIMARY KEY (student_id)
);`

`CREATE TABLE enrollment (
  student_id INTEGER,
  course_id INTEGER,
  term TEXT,
  year INTEGER
);`

`CREATE TABLE course (
  course_id INTEGER,
  name TEXT,
  desc TEXT,
  PRIMARY KEY (course_id)
);`

#### [Data types in SQLite](https://www.sqlite.org/datatype3.html)
There are 5 data types for values stored in in an SQLite database:

- NULL. The value is a NULL value.

- INTEGER. The value is a signed integer, stored in 1, 2, 3, 4, 6, or 8 bytes depending on the magnitude of the value.

- REAL. The value is a floating point value, stored as an 8-byte IEEE floating point number.

- TEXT. The value is a text string, stored using the database encoding (UTF-8, UTF-16BE or UTF-16LE).

- BLOB. The value is a blob of data, stored exactly as it was input.

## Create database connection
First, we will create a database connection and a cursor to execute SQL commands.

In [110]:
conn = sqlite3.connect(gdrive+'student.db') # creates a connection to the database
c = conn.cursor() # the cursor is what we'll actually use

Execute a SQL command to create a `student` table with 3 columns `student_id`, `firstname`, `lastname`.
Here we set the `id` as a primary key.

In [111]:
### Let's create a student table
SQL_CreateTable = '''CREATE TABLE <name of entity>(
             student_id <datatype>,
             firstname <datatype>,
             lastname <datatype>
             )'''

c.execute(SQL_CreateTable)

In [158]:
### Add data to table

data = [
    (1013832, 'Martin', 'Jones'),
    (1284729, 'Wendy', 'Lee'),
    (1286189, 'Clint', 'Eastwood'),
    (1014632, 'Benito', 'Antonio'),
    (1357037, 'Kim', 'Namjoon'),
    (4927502, 'Ludwig van', 'Beethoven')
]

c.executemany("<task> <entity> VALUES (?, ?, ?);", <data>)

# ## Another way to do the same step
# SQL_InsertStmt = """INSERT INTO student VALUES
#                  (1013832, 'Martin', 'Jones'),
#                  (1284729, 'Wendy', 'Lee')
#                  (1286189, 'Clint', 'Eastwood'),
#                  (1250932, 'Benito', 'Antonio'),
#                  (1357037, 'Kim', 'Namjoon'),
#                  (4927502, 'Ludwig van', 'Beethoven')
#                  """

# c.execute(SQL_InsertStmt)

In [159]:
### Check what's in the table
c.execute("SELECT * FROM student;")
print(c.fetchall())

[(1013832, 'Martin', 'Jones'), (1014632, 'Benito', 'Antonio'), (1284729, 'Wendy', 'Lee'), (1286189, 'Clint', 'Eastwood'), (1357037, 'Kim', 'Namjoon'), (2584563, 'Theodore', 'Roosevelt'), (4927502, '', 'Pitt')]


In [160]:
### Fetch a particular entry
c.execute(<fetching one entry from student>)
results = c.fetchall()
print(results)

[(1013832, 'Martin', 'Jones')]


In [161]:
### Insert option
## INSERT OR IGNORE: This will add a new entry if not duplicate. If there is a duplicate it will skip
c.execute("INSERT OR IGNORE INTO student VALUES (?, ?, ?)", (2584563, 'Teddy', 'Roosevelt'))

## INSERT OR REPLACE: This will overwrite existing row with same student_id
c.execute()

## View the current entity
c.execute("SELECT * FROM student;")
print(c.fetchall())

[(1013832, 'Martin', 'Jones'), (1014632, 'Benito', 'Antonio'), (1284729, 'Wendy', 'Lee'), (1286189, 'Clint', 'Eastwood'), (1357037, 'Kim', 'Namjoon'), (2584563, 'Theodore', 'Roosevelt'), (4927502, '', 'Pitt')]


#### Let's create the other two tables: `course` and `enrollment`.

In [117]:
SQL_CreateTable = '''CREATE TABLE IF NOT EXISTS course (
             course_id
             name
             desc
             )'''

c.execute(SQL_CreateTable)

In [118]:
### In SQLite, foreign keys are OFF by default. Need to enable them
# c.execute("PRAGMA foreign_keys = ON;")

SQL_CreateTable = '''CREATE TABLE IF NOT EXISTS enrollment (
             student_id INTEGER,
             course_id INTEGER,
             term TEXT NOT NULL,
             year INTEGER,
             FOREIGN KEY (student_id)

             FOREIGN KEY (course_id)

             )'''

c.execute(SQL_CreateTable)

In [98]:
# c.execute("SELECT * FROM enrollment;")
# print(c.fetchall())

`DELETE CASCADE`: When we create a foreign key using this option, it deletes the referencing rows in the child table when the referenced row is deleted in the parent table which has a primary key.

`UPDATE NO ACTION`: When we create a foreign key using this option, the foreign key constraint (NO ACTION) behaves whenever the parent key is updated.

Read more about [foreign keys](https://www.sqlite.org/foreignkeys.html) in SQLite.

## Load CSV file into sqlite table

In [136]:
import pandas as pd

# load the course data into panda dataframe
course = pd.read_csv("https://raw.githubusercontent.com/CS22B-IntroPython/CS22B_SP26_datafiles/refs/heads/main/Module08_sqlite/course.csv")
course

,course_id,name,desc
0,28732,CS122,Advanced Python
1,24852,CS22B,Intro to Python II
2,37473,CS22A,Intro to Python I


In [137]:
### Insert the data from dataframe to database table `course` using 'append'
## if already exist, use 'replace'
course.to_sql(<entity>,<cursor>,<if_exists=>, index=False)

3

In [170]:
### Use "SELECT * FROM to check results

In [139]:
# load the enrollment data into panda dataframe
edf = pd.read_csv("https://raw.githubusercontent.com/CS22B-IntroPython/CS22B_SP26_datafiles/refs/heads/main/Module08_sqlite/enroll.csv")
edf

,student_id,course_id,term,year
0,2156349,24852,Fall,2020
1,1250932,28732,Spring,2021
2,1357037,24832,Fall,2020
3,4927502,28752,Fall,2021
4,1013832,37473,Fall,2020
5,1284729,24852,Spring,2021


In [140]:
### Insert the data from dataframe to database table `enrollment`
edf.to_sql('enrollment', conn, if_exists='replace', index=False)

6

In [124]:
### Use "SELECT * FROM to check results
c.execute()


[(2156349, 24852, 'Fall', 2020), (1250932, 28732, 'Spring', 2021), (1357037, 24832, 'Fall', 2020), (4927502, 28752, 'Fall', 2021)]


In [141]:
# Can read directly from a SQL Query to a pandas dataframe
# https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html
pd.read_sql_query("SELECT * FROM enrollment", conn)

,student_id,course_id,term,year
0,2156349,24852,Fall,2020
1,1250932,28732,Spring,2021
2,1357037,24832,Fall,2020
3,4927502,28752,Fall,2021
4,1013832,37473,Fall,2020
5,1284729,24852,Spring,2021


In [164]:
SQL_JointQuery = """SELECT student.firstname, student.lastname,
                course.name FROM student
                JOIN enrollment using (student_id)
                JOIN course using (course_id)
                WHERE student.lastname='Lee'
                """
c.execute(SQL_JointQuery)
joint_results = c.fetchall()
print(joint_results)

[('Wendy', 'Lee', 'CS22B')]


In [165]:
### Does this student exist?
c.execute("SELECT * FROM student")
# c.execute("SELECT * FROM student WHERE lastname='Namjoon';")
print(c.fetchall())

[(1013832, 'Martin', 'Jones'), (1014632, 'Benito', 'Antonio'), (1284729, 'Wendy', 'Lee'), (1286189, 'Clint', 'Eastwood'), (1357037, 'Kim', 'Namjoon'), (2584563, 'Theodore', 'Roosevelt'), (4927502, '', 'Pitt')]


In [167]:
# Another way to join tables
SQL_JointQuery2 = """SELECT student.firstname, student.lastname,
                course.name FROM student
                JOIN enrollment ON enrollment.student_id = student.student_id
                JOIN course ON enrollment.course_id = course.course_id
                WHERE student.lastname='Lee'
                """
c.execute(SQL_JointQuery2)
joint_results2 = c.fetchall()
print(joint_results2)

[('Wendy', 'Lee', 'CS22B')]


To save the changes, commit the current transaction using `conn.commit()`. The cursor makes changes, and connection commits. To close the connection with conn.close()`.